# 05 — Topic Interpretation (Student A)
**BSAN 6200 Assignment 3 | Disneyland Reviews**

Student A interprets Topics 1–5 for all three parks.

Process followed for each topic:
1. Read top 20 words from model output
2. Read 5–8 representative documents (highest topic-probability reviews)
3. Identified recurring patterns and sentiment
4. Drafted topic name

## Setup — Load Models

In [5]:
import warnings
warnings.filterwarnings('ignore')
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.read_csv('data/processed/disneyland_reviews_processed.csv')
target_groups  = ['California', 'Paris', 'Hong Kong']
selected_k_lda = {'California': 10, 'Paris': 10, 'Hong Kong': 10}

def topic_word_table(components, feature_names, top_n=20):
    rows = []
    for idx, topic in enumerate(components):
        top_ids   = topic.argsort()[-top_n:][::-1]
        top_words = [feature_names[i] for i in top_ids]
        rows.append({'topic_id': idx+1, 'top_words': ', '.join(top_words)})
    return pd.DataFrame(rows)

def top_docs_for_each_topic(subset_df, doc_topic_matrix, top_n_docs=8):
    rows = []
    for t in range(doc_topic_matrix.shape[1]):
        probs = doc_topic_matrix[:, t]
        for rank, doc_id in enumerate(np.argsort(probs)[::-1][:top_n_docs], 1):
            rows.append({'topic_id': t+1, 'rank': rank,
                         'probability': float(probs[doc_id]),
                         'text': subset_df.iloc[doc_id]['review_text'][:800]})
    return pd.DataFrame(rows)

def topic_prevalence_table(doc_topic_matrix):
    avg_probs = doc_topic_matrix.mean(axis=0)
    return pd.DataFrame({'topic_id': range(1, len(avg_probs)+1),
                         'avg_topic_probability': avg_probs})

lda_topic_word_tables = {}
lda_top_doc_tables    = {}
lda_prevalence_tables = {}

for group in target_groups:
    k    = selected_k_lda[group]
    safe = group.lower().replace(' ', '_')
    with open(f'models/lda_{safe}_k{k}.pkl', 'rb') as f:
        fitted = pickle.load(f)
    subset = df[df['location_clean'] == group].copy().reset_index(drop=True)
    lda_topic_word_tables[group] = topic_word_table(fitted['model'].components_, fitted['feature_names'])
    lda_top_doc_tables[group]    = top_docs_for_each_topic(subset, fitted['doc_topic'])
    lda_prevalence_tables[group] = topic_prevalence_table(fitted['doc_topic'])
    print(f'{group}: loaded k={k} model ✓')

California: loaded k=10 model ✓
Paris: loaded k=10 model ✓
Hong Kong: loaded k=10 model ✓


## Reference: Top 20 Words Per Topic

In [6]:
for group in target_groups:
    print(f'\n{"="*60}')
    print(f'  {group} — top 20 words per topic')
    print(f'{"="*60}')
    display(lda_topic_word_tables[group])


  California — top 20 words per topic


,topic_id,top_words
0,1,"year, always, love, earth, happiest, every, vi..."
1,2,"pas, fast, ticket, pass, early, use, wait, hou..."
2,3,"character, mickey, halloween, photo, birthday,..."
3,4,"mountain, pirate, california, adventure, space..."
4,5,"world, florida, magic, california, better, vis..."
5,6,"kid, fun, year, child, old, loved, family, wou..."
6,7,"food, hotel, restaurant, water, kid, eat, take..."
7,8,"wait, line, long, minute, mountain, crowd, fas..."
8,9,"would, hour, could, crowd, many, ticket, wait,..."
9,10,"parade, firework, show, night, christmas, ligh..."



  Paris — top 20 words per topic


,topic_id,top_words
0,1,"ticket, fast, pas, take, train, paris, hour, q..."
1,2,"queue, closed, staff, would, hour, smoking, co..."
2,3,"queue, character, child, parade, kid, year, wo..."
3,4,"food, expensive, attraction, price, queue, kid..."
4,5,"staff, christmas, child, year, would, characte..."
5,6,"show, amazing, parade, firework, castle, magic..."
6,7,"mountain, space, thunder, pirate, big, coaster..."
7,8,"paris, florida, orlando, world, would, french,..."
8,9,"hotel, stayed, take, room, food, breakfast, bu..."
9,10,"food, restaurant, meal, eat, mickey, euro, cha..."



  Hong Kong — top 20 words per topic


,topic_id,top_words
0,1,"show, parade, firework, kid, visit, experience..."
1,2,"mountain, space, land, mystic, toy, grizzly, s..."
2,3,"hot, weather, rain, summer, visit, parade, hea..."
3,4,"character, show, parade, mickey, photo, take, ..."
4,5,"hotel, food, restaurant, night, stayed, room, ..."
5,6,"kong, hong, small, smaller, world, paris, well..."
6,7,"train, mtr, easy, hong, kong, station, ticket,..."
7,8,"kid, old, year, child, would, young, adult, lo..."
8,9,"food, ticket, expensive, water, buy, bring, pr..."
9,10,"queue, visit, attraction, long, hong, kong, wa..."


## Reference: Representative Documents (Topics 1–5)

In [7]:
for group in target_groups:
    print(f'\n{"="*60}')
    print(f'  {group} — representative docs for Topics 1–5')
    print(f'{"="*60}')
    docs = lda_top_doc_tables[group][lda_top_doc_tables[group]['topic_id'] <= 5]
    for _, row in docs.iterrows():
        print(f'\n--- Topic {row["topic_id"]} | Rank {row["rank"]} | prob={row["probability"]:.3f} ---')
        print(row['text'])


  California — representative docs for Topics 1–5

--- Topic 1 | Rank 1 | prob=0.972 ---
Once in the park you seem to get caught up in the excitement of the surroundings.No matter how old you become its still just as exciting as the first time you visit. The only problem is that the prices seem to rise every single year so it gets kinda hard to visit anymore. yes there a monthly payment plan its still the same cost no matter how long you take to pay it off.

--- Topic 1 | Rank 2 | prob=0.972 ---
Im 23, but there is no place like Disneyland. I now liv far away but recently was able to go visit family and take trip to the happiest place on earth. I am always pleased with the treatment I recieve from every single cast member. I know how hard they work to make DL a place everyone wants to be and will always remember. I suggest taking all kids to visit at least once and let them experience the magic!

--- Topic 1 | Rank 3 | prob=0.971 ---
The happiest place on earth, the happiest place on 

## Reference: Topic Prevalence

In [8]:
for group in target_groups:
    print(f'\n{"="*60}')
    print(f'  {group} — topic prevalence')
    print(f'{"="*60}')
    display(lda_prevalence_tables[group].sort_values('avg_topic_probability', ascending=False))


  California — topic prevalence


,topic_id,avg_topic_probability
5,6,0.174258
0,1,0.137037
1,2,0.126297
8,9,0.115056
7,8,0.106106
4,5,0.085811
6,7,0.078173
9,10,0.065390
3,4,0.064206
2,3,0.047665



  Paris — topic prevalence


,topic_id,avg_topic_probability
2,3,0.184880
3,4,0.125332
5,6,0.124076
1,2,0.116642
0,1,0.091619
8,9,0.084253
7,8,0.081628
4,5,0.079057
6,7,0.068786
9,10,0.043726



  Hong Kong — topic prevalence


,topic_id,avg_topic_probability
0,1,0.189034
7,8,0.132392
5,6,0.131061
9,10,0.122212
1,2,0.088490
8,9,0.086095
3,4,0.083078
6,7,0.070598
2,3,0.054747
4,5,0.042293


## Topic Interpretations — Student A (Topics 1–5)

### California



**Topic 1 — Practical Visit Tips & Logistics Planning** *(Prevalence: 9.8%)*

Top words: `hotel, take, water, around, parade, bring, kid, early, bus, locker`

Reviews in this topic are written as visitor guides — experienced visitors sharing specific, actionable advice for navigating the park. Recurring themes include bringing your own food and using lockers at the entrance to save money, arriving early to beat queues, and using Fastpass for popular rides. Sentiment is mixed but informative rather than strongly positive or negative. In the representative documents, reviewers specifically mention using lockers to bring sandwiches and a cooler, finding a hidden rest area near the tram drop-off with tables and water fountains, arriving 15 minutes before rides open, and bringing bandaids and blister protection for the long walking day.


**Topic 2 — Food Quality, Dining Venues & Culinary Highlights** *(Prevalence: 9.4%)*

Top words: `food, restaurant, character, experience, eat, blue, bayou, main, street, tour`

This topic clusters reviews centred on specific dining experiences inside the park — named restaurants, particular dishes, and food quality comparisons. Sentiment is mixed: some reviewers celebrate standout dishes and venues, while others mourn discontinued menu items or criticise declining quality. In the representative documents, reviewers specifically name Blue Bayou (praising the Monte Cristo sandwich and plantation veranda setting), Cafe Orleans, Plaza Inn fried chicken on Main Street, Trader Sam's Tiki Bar near the original Disneyland Hotel, and the Napa Rose inside the hotel as first-rate. One reviewer lists over a dozen discontinued dishes by name across multiple restaurants.


**Topic 3 — Overcrowding, Ticket Prices & Value for Money** *(Prevalence: 11.8%)*

Top words: `would, year, price, many, crowd, money, ticket, cost, long, line`

This topic captures visitor frustration with the gap between what Disneyland costs and what the experience delivers when the park is at capacity. Annual passholders feature heavily, comparing current crowded conditions to better past visits. Sentiment is predominantly negative. In the representative documents, reviewers describe travelling from Australia only to find the experience a deep disappointment, paying for premium upgrades that made no difference to wait times, simultaneously finding 3–4 major attractions closed for maintenance, and seriously considering not renewing annual passes after particularly crowded visits.


**Topic 4 — Night Shows, Fireworks & Parade Spectacles** *(Prevalence: 9.8%)*

Top words: `parade, firework, show, night, amazing, fun, watch, fantasmic, paint, world`

Reviews in this topic focus on evening entertainment — the Paint the Night parade, Fantasmic water show, and fireworks display. First-time visitors dominate and use superlative language throughout. Sentiment is overwhelmingly positive. In the representative documents, reviewers describe the Paint the Night parade floats as mind-blowing with extravagant illuminated characters, the fireworks as breathtaking with synchronised movie clips, and Fantasmic as spectacular. Several reviewers specifically note the scheduling conflict between Fantasmic and the night parade, and multiple reviewers recommend staying in the park until close to catch both shows.


**Topic 5 — Family Milestones & Children's First Disney Visits** *(Prevalence: 12.1%)*

Top words: `year, kid, old, year old, loved, love, every, family, first, time`

This is the most prevalent topic for California and captures emotionally resonant reviews from parents bringing young children for the first time, or adults returning after many years away. Reviews frequently express a strong generational connection to the park. Sentiment is strongly positive and nostalgic. In the representative documents, reviewers describe 2- and 3-year-olds who haven't stopped talking about the trip since returning home, a teenager who requested a return trip as her high school graduation gift, parents who felt like children again themselves, and families who have made annual trips for six or seven consecutive years.

### Paris


**Topic 1 — Ticket Purchasing, Transport & Pre-Visit Planning** *(Prevalence: 8.5%)*

Top words: `ticket, pas, paris, fast, studio, fast pas, train, rer, online, book`

This topic groups reviews that advise on how to get to and into the park — booking tickets online, navigating the RER A train from central Paris, managing two-park passes, and using Fast Pass. Sentiment is practical and informative. In the representative documents, reviewers note that online tickets cost approximately €47 per person versus €90 at the gate, that the RER A takes about 1 hour 20 minutes from central Paris, and that Walt Disney Studios closes at 19:00 while the main Disneyland Park closes at 23:00. One reviewer details the disability pass process, which requires a doctor's letter and proof of DLA to obtain a Disney green pass from City Hall inside the park.



**Topic 2 — Overpriced Food, Drink & Unreliable Fastpass System** *(Prevalence: 12.2%)*

Top words: `food, queue, euro, expensive, drink, fast, hour, water, bring, snack`

Reviews in this topic share frustration with the cost of food and drink inside the park and the unreliability of the Fastpass system. Visitors frequently advise future visitors to bring their own food and fill up water bottles at the park's water fountains. Sentiment is strongly negative. In the representative documents, reviewers describe beer costing €8 per pint, kids' Coke at €3.75, Fastpass machines breaking down with queues of 100 people, and waiting 40 minutes at a so-called fast food counter. One reviewer explicitly tells future visitors to "save your money and take your kids to the local rec" rather than pay the park's food prices.



**Topic 3 — Staff Attitude, Ride Closures & Park Deterioration** *(Prevalence: 11.6%)*

Top words: `staff, would, closed, member, paris, year, could, rude, attraction, refurbishment`

This topic captures reviews dominated by disappointment with staff behaviour and a high frequency of attraction closures. Paris is repeatedly and unfavourably compared to US Disney parks in terms of service standards and park upkeep. Sentiment is strongly negative. In the representative documents, reviewers describe French cast members as surly and dismissive of English speakers, finding 15–20% of attractions closed for refurbishment during a single visit, peeling paint, cigarette smoke in non-smoking areas, and overflowing bins. One reviewer emailed Disney directly about their experience and received no reply, calling it unprofessional.


**Topic 4 — Queue Wait Times & Crowd Management for Families** *(Prevalence: 10.2%)*

Top words: `attraction, child, kid, queue, minute, waiting, hour, long, busy, summer`

Reviews in this topic focus on wait times and how they affect family visits — particularly parents with young children struggling with long queues during peak summer season. Sentiment ranges from resigned acceptance to outright frustration. In the representative documents, reviewers describe spending 8 hours in the park and completing only 3 rides, facing 90-minute queues for single attractions, and one reviewer who proposed implementing digital queue numbers rather than requiring visitors to stand in physical lines. Several reviewers specifically recommend visiting in off-peak months such as January, February, or November to avoid the worst crowds.


**Topic 5 — Paris vs US Disney Parks: Direct Comparisons** *(Prevalence: 8.9%)*

Top words: `paris, florida, world, orlando, magic, attraction, compare, castle, beautiful, visit`

This topic groups reviews that explicitly compare Disneyland Paris to Walt Disney World Orlando or Disneyland California. Sentiment is split — some visitors are pleasantly surprised by Paris; others use the comparison to discourage future visits. In the representative documents, reviewers praise the Paris castle's stained-glass windows and its hidden animatronic dragon beneath Sleeping Beauty's castle as superior to California. Others criticise Paris for outdated attractions and inferior cleanliness compared to US parks. One reviewer who has visited US Disney parks over 10 times describes Paris as the first time they genuinely felt the Disney magic.

### Hong Kong


**Topic 1 — Park Scale & Managing Visitor Expectations** *(Prevalence: 12.9%)*

Top words: `hong, kong, hong kong, smaller, small, visit, compare, tokyo, world, day`

This topic captures reviews where visitors explicitly reflect on Hong Kong Disneyland's smaller scale relative to other Disney parks worldwide. Reviewers are largely managing expectations for others rather than expressing strong emotion. Sentiment is measured and factual. In the representative documents, reviewers who have visited Tokyo, Orlando, Anaheim, and Paris all describe Hong Kong as the smallest of the Disney parks, note they were able to walk straight onto rides due to low crowds, and recommend the park primarily for families with young children or as a day trip. One reviewer states it should not be the top item on a Hong Kong itinerary but is worth fitting in if time allows.



**Topic 2 — Overpriced Food & Bag-Check Restrictions** *(Prevalence: 7.3%)*

Top words: `food, water, expensive, bring, drink, inside, bottle, price, outside, snack`

Reviews in this topic focus on the high cost of food and drink combined with bag checks at the entrance that prevent visitors bringing their own food inside. Sentiment is negative specifically around pricing, though overall park enjoyment is often still rated positively. In the representative documents, reviewers describe paying $5 AUD for a single bottle of water, being advised to bring their own water bottle to fill at the park's fountains, finding many food stalls closed and having to walk a long distance to find refreshments, and vegetarian visitors being frustrated by very limited food options. One reviewer describes Disney as having visitors "over a barrel" because bag checks prevent bringing outside food while internal options are expensive and limited.



**Topic 3 —  General Park Operations, Staff & Queue Management** *(Prevalence: 14.4%)*

Top words: `staff, would, food, queue, small, well, attraction`

This is the most prevalent topic for Hong Kong and captures a recurring and specific complaint about crowd behaviour — particularly perceived queue-cutting and pushing by visitors from mainland China. While the top words are generic, the representative documents consistently and explicitly describe this pattern. Sentiment is strongly negative. In the representative documents, reviewers describe politely observing queue-jumping behaviour that would not be tolerated at Anaheim, staff appearing unable or unwilling to manage the situation, and the problem being worst during mainland China public holiday periods. One reviewer explicitly advises European visitors to go to Paris instead because of this issue.



**Topic 4 — Character Meet-and-Greet Queues & Photo Opportunities** *(Prevalence: 9.7%)*

Top words: `queue, ticket, photo, take, pas, character, arrive, early, long, wait`

Reviews in this topic focus on interactions with Disney characters — specifically the frustration of long queues for photo opportunities and limited character availability walking freely through the park. Sentiment is mixed: disappointment about character access is common, but the parade is consistently cited as a positive fallback. In the representative documents, reviewers describe joining a character photo queue only to abandon it after waiting too long with no visible end in sight, seeing characters only briefly during the parade rather than walking freely through the park, and confusion around third-party ticket purchases causing 30-minute delays at Guest Relations. One reviewer states that without characters walking freely around the park, Disneyland loses its sense of magic.



**Topic 5 — Family-Friendly Experience & Suitability for Young Kids** *(Prevalence: 14.4%)*

Top words: `kid, child, visit, adult, young, family, small, recommend, great, enjoy`

This topic captures broadly positive reviews from families who found Hong Kong Disneyland well-suited to their needs — particularly for younger children. Reviewers frequently position the park as ideal for a specific audience and offer practical recommendations. Sentiment is positive and practical. In the representative documents, reviewers describe riding every ride multiple times due to short queues on weekday visits, recommend visiting in winter for cooler and less crowded conditions, and explicitly position the park as ideal for children aged 3 to 9 or for families who have never visited a Disney park before. Several reviewers who have also visited larger Disney parks note that Hong Kong is the right size for a manageable one-day family outing.